# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aabdullahhtar-create/flyrank-ML-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Lane:** Refresh / Content Opportunity Scoring.

I frame this primarily as a **ranking / scoring task**. The decision is not simply “is this page declining?”; it is **which pages should a content editor review first** when review time is limited. The system should assign each eligible content item a priority score and sort the items into a review queue.

The output supports a real action: an editor reviews the highest-priority pages first and then decides whether to refresh, expand, protect, prune, or monitor them. The score is decision support, not an automatic instruction to edit a page.

For the starter-data exercise I use pages with enough activity to make the opportunity measurable: at least 100 impressions in the trailing 90 days and at least one session. This is a provisional lane slice, not a claim that pages outside it have no value.

In [1]:
from pathlib import Path
import pandas as pd

# Work both in Colab (repo root) and when executed from work/notebooks locally.
candidates = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]
DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not find data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)
lane = df[(df["impressions_90d"] >= 100) & (df["sessions_90d"] > 0)].copy()

print(f"Starter rows: {len(df):,}")
print(f"Lane slice: {len(lane):,} pages ({len(lane)/len(df):.1%} of starter data)")


Starter rows: 30,000
Lane slice: 22,006 pages (73.4% of starter data)


## 2. Target or proxy

For **ranking**, the ideal target is a later, observed outcome that tells us whether a page actually became a worthwhile review opportunity. A stronger capstone target would therefore be built from the warehouse's time series: use an earlier window for features and a **future window** for an observed outcome such as meaningful search-impression decline or recovery opportunity.

The starter CSV is only a trailing-90-day snapshot. It contains `trend_direction`, and the repo's teaching pipeline defines a declining label from it. I use that only as a **provisional proxy for this framing exercise**:

`decline_proxy = 1` when `trend_direction == "down"`, otherwise `0`.

This proxy is rule-defined from `trend_pct`, so it is **not an independent future truth label**. A model trained to reproduce it could mainly learn the existing rule. Also, `trend_direction` and `trend_pct` must never be model features when this proxy is the target. The later warehouse version should replace this proxy with a properly separated future-window outcome.

The score I ultimately want is a **review-priority score**, combining evidence about risk/opportunity so pages can be ranked rather than automatically classified as “refresh” or “do not refresh.”

In [2]:
lane["decline_proxy"] = (lane["trend_direction"] == "down").astype(int)

proxy_counts = (
    lane["decline_proxy"]
    .value_counts()
    .rename(index={0: "not_down", 1: "down"})
    .to_frame("pages")
)
proxy_counts["share"] = proxy_counts["pages"] / len(lane)
display(proxy_counts)

print(
    "Important: this proxy comes from trend_direction/trend_pct; "
    "those columns would be excluded from model features."
)


,pages,share
decline_proxy,,
down,13152,0.597655
not_down,8854,0.402345


Important: this proxy comes from trend_direction/trend_pct; those columns would be excluded from model features.


## 3. Success metric

My primary success metric is **Precision@20**: among the 20 pages placed at the top of the review queue, what fraction are positive according to the evaluation target?

I choose this because the product decision is a ranked, capacity-limited editor queue. A high Precision@20 means scarce review time is concentrated on pages that the evaluation outcome says deserved attention. It is more aligned with the action than overall accuracy.

For this starter proxy, the naive reference rate among eligible pages is about 60% declining, so a ranking should do meaningfully better than simply selecting pages without prioritization. I will not declare a final “good” threshold from this snapshot alone. Before model training, I would compare Precision@20 against a transparent rule baseline and then validate on held-out clients / a future time window. I would also inspect the top 20 by hand because a high metric does not prove that refreshing those pages will cause improvement.

In [3]:
base_rate = lane["decline_proxy"].mean()
k = 20

print(f"Eligible-page decline-proxy base rate: {base_rate:.1%}")
print(f"Primary metric: Precision@{k}")
print(
    "Reference expectation: an uninformative/random ranking would average "
    f"about {base_rate:.1%} positives in its top {k} over repeated samples."
)


Eligible-page decline-proxy base rate: 59.8%
Primary metric: Precision@20
Reference expectation: an uninformative/random ranking would average about 59.8% positives in its top 20 over repeated samples.


## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized content item (page).**

The starter file has one row per content item. For this lane I keep the measurable-opportunity slice (`impressions_90d >= 100` and `sessions_90d > 0`). The dataframe below shows identifiers for tracing/grouping plus candidate context signals and the provisional proxy.

`content_id` and `client_id` are **not model features**. `client_id` is useful for grouped validation so pages from the same client do not casually leak across train/test splits. Rates are stored as ×100 percentages, so for example `ctr = 0.76` means 0.76%, not 76%. Also, `avg_position = 0` means no position data.

In [4]:
unit_columns = [
    "content_id",
    "client_id",
    "content_type",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "engagement_rate",
    "decline_proxy",
]

unit_df = lane[unit_columns].copy()
print(f"Shape of lane dataframe: {unit_df.shape}")
print("Grain check — duplicated content_id rows:", unit_df["content_id"].duplicated().sum())
display(unit_df.head(10))


Shape of lane dataframe: (22006, 12)
Grain check — duplicated content_id rows: 0


,content_id,client_id,content_type,impressions_90d,clicks_90d,sessions_90d,ctr,avg_position,content_age_days,days_since_last_update,engagement_rate,decline_proxy
0,content_304f48230142,client_f369cb89fc,keyword article,3803,29,17,0.76,10.6,187,20,5.88,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,7,9,0.05,20.3,445,25,0.00,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,11,0.09,36.5,141,20,0.00,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,58,78,0.49,6.2,463,22,1.28,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,24,145,0.13,44.0,263,14,0.00,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,3970,1,5,0.03,8.5,147,20,0.00,1
7,content_a63219c6e95a,client_19581e27de,keyword article,1724,1,28,0.06,21.2,445,22,3.57,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,32574,29,68,0.09,46.0,90,20,5.88,1
9,content_c27558df2b0c,client_19581e27de,keyword article,1240,2,3,0.16,4.9,257,104,0.00,1
10,content_d8ee6cc6d642,client_19581e27de,keyword article,20919,324,326,1.55,2.2,329,104,6.75,0


## 5. Why ML beats a fixed rule here

A fixed rule is an important **baseline**, but it is unlikely to be the final ranking method because review priority can depend on several signals at once: visibility, position, clicks/CTR, engagement, content age, time since update, demand context, and content type. Their relationships can interact. For example, the same CTR may mean something different for a highly visible page than for a page with very few impressions, and missingness itself follows content type.

ML earns a place only if it can rank useful review candidates **better than a transparent rule baseline on held-out data**. If it cannot, I should keep the simpler rule.

The intended decision loop is:

**page measurements → priority score/rank → editor reviews top pages → editor chooses an action**

This remains observational decision support. A high score does **not** mean a refresh will cause traffic growth, and a decline proxy does not prove that a page should be edited.

In [5]:
# A small check showing why one threshold alone does not describe the lane.
# These are descriptive groups, not a model and not causal evidence.
summary = (
    lane.assign(
        stale_90d=lane["days_since_last_update"] >= 90,
        page_one=(lane["avg_position"] > 0) & (lane["avg_position"] <= 10),
        has_clicks=lane["clicks_90d"] > 0,
    )
    .groupby(["stale_90d", "page_one"], dropna=False)
    .agg(
        pages=("content_id", "size"),
        decline_proxy_rate=("decline_proxy", "mean"),
        median_impressions=("impressions_90d", "median"),
        median_ctr_pct=("ctr", "median"),
    )
    .reset_index()
)

summary["decline_proxy_rate"] = (summary["decline_proxy_rate"] * 100).round(1)
display(summary)

print(
    "Different signal combinations have different measured profiles. "
    "This motivates testing a multivariable ranking method against a simple rule baseline."
)


,stale_90d,page_one,pages,decline_proxy_rate,median_impressions,median_ctr_pct
0,False,False,7857,57.0,971.0,0.09
1,False,True,6030,60.0,2734.0,0.24
2,True,False,4934,61.0,1733.0,0.09
3,True,True,3185,64.4,3420.0,0.20


Different signal combinations have different measured profiles. This motivates testing a multivariable ranking method against a simple rule baseline.


## Self-check

Before submission:

- [x] Every section above is filled — markdown thinking AND code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries are included; identifiers are pseudonymous
- [x] Claims use careful words such as observed, measured, proxy, directional, and decision-support
- [x] ML task type is named: ranking / scoring
- [x] Target/proxy and its limitation are explicit
- [x] Success metric is named before training: Precision@20
- [x] Unit of analysis is shown as a real dataframe: one row = one content item/page
- [x] The output is tied to a real content action: an editor reviews the ranked queue
- [x] I explain that ML must beat a transparent fixed-rule baseline to earn its complexity
- [ ] Commit this executed notebook to `work/notebooks/w02_ml_task_framing.ipynb`, then submit the public repo URL
